# Sprint 1 - Minimum Viable Pipeline

**Goal (see AGENT.md / final_brief_and_plan.md):** get one end-to-end training run on PlantVillage.

- Load MobileNetV2 pretrained on ImageNet, head swapped to our 38 classes
- Stage 1 only: base frozen, train just the classification head - no augmentation yet
- Evaluate on the PlantVillage held-out test set and log metrics
- **Done when:** `python -m ml.predict --checkpoint ... --image ...` prints a class + confidence

**Where data comes from:** the Drive archives from `sprint0_data_exploration.ipynb`. This notebook
hydrates + organizes locally itself, so it's self-contained after Sprint 0's durability gate passed.

**Artifacts go to Drive** (a handful of files, unlike Sprint 0's many):
`folium/checkpoints/` (one checkpoint per epoch + `best_*`), `folium/results/ablation_results.csv`
(the paper's Results table source of truth) and `cm_baseline_pv_only_no_aug.png`.

In [ ]:
import platform
import subprocess
import sys

print("Python:", sys.version.split()[0])
print("Platform:", platform.platform())
try:
    gpu = subprocess.run(["nvidia-smi"], capture_output=True, text=True, timeout=30)
    print(gpu.stdout.strip().splitlines()[0] if gpu.stdout.strip() else gpu.stderr.strip() or "No GPU detected (CPU only)")
except Exception as exc:
    print("GPU check skipped:", exc)

## Step 1 - Mount Drive + clone repo

Requires the Sprint 0 archives (`plantvillage_raw.zip` + manifest) to be on Drive and verified.

In [ ]:
from google.colab import drive
drive.mount("/content/drive")

from pathlib import Path

REPO_URL = "https://github.com/io-PEAK/folium.git"   # change if you forked
REPO_DIR = Path("/content/folium")
DATA_DIR = Path("/content/drive/MyDrive/folium/data")    # durable archives (from Sprint 0)
LOCAL_RAW_DIR = Path("/content/folium_raw")             # per-session raw
LOCAL_DATA_DIR = Path("/content/folium_data")            # per-session organized splits
CHECKPOINT_DIR = Path("/content/drive/MyDrive/folium/checkpoints")
RESULTS_DIR = Path("/content/drive/MyDrive/folium/results")

if not (REPO_DIR / "ml").exists():
    %cd /content
    !git clone --depth 1 {REPO_URL}
else:
    !git -C {REPO_DIR} pull --ff-only -q

for d in (DATA_DIR, LOCAL_RAW_DIR, LOCAL_DATA_DIR, CHECKPOINT_DIR, RESULTS_DIR):
    d.mkdir(parents=True, exist_ok=True)
print("DATA_DIR (archives):", DATA_DIR)
print("LOCAL_DATA_DIR:", LOCAL_DATA_DIR)
print("CHECKPOINT_DIR:", CHECKPOINT_DIR)
print("RESULTS_DIR:", RESULTS_DIR)

## Step 2 - Install dependencies

In [ ]:
%pip install -q --upgrade pip
%pip install -q torch torchvision albumentations matplotlib pandas tqdm opencv-python-headless scikit-learn

## Step 3 - Hydrate raw from Drive, then organize splits locally

Same as Sprint 0: unzip the two archives from Drive into local raw, then build
`train/val/test` folders with `scripts/organize_datasets.py`. Deterministic and
idempotent, so re-running is harmless.

In [ ]:
import sys

sys.path.insert(0, str(REPO_DIR))
from scripts.download_datasets import PLANTVILLAGE_EXPECTED, PLANTDOC_EXPECTED, hydrate_dataset

for name, expected in (("plantvillage", PLANTVILLAGE_EXPECTED), ("plantdoc", PLANTDOC_EXPECTED)):
    try:
        hydrate_dataset(LOCAL_RAW_DIR, DATA_DIR, name, expected)
    except RuntimeError as exc:
        print("HYDRATE FAILED:", exc)
        raise

result = subprocess.run([
    sys.executable,
    str(REPO_DIR / "scripts" / "organize_datasets.py"),
    "--raw-dir", str(LOCAL_RAW_DIR),
    "--data-dir", str(LOCAL_DATA_DIR),
], cwd=str(REPO_DIR))
assert result.returncode == 0, "organize_datasets.py failed"
print("splits ready at", LOCAL_DATA_DIR)

## Step 4 - Train (Stage 1: frozen backbone, head only, no augmentation)

`python -m ml.train` on the PlantVillage train/val splits. Saves a checkpoint
per epoch + `best_plantvillage_stage1.pt` directly to Drive so a dropped Colab
session never loses the run. Defaults: 5 epochs, batch 32, Adam lr 1e-3, seed 42.

> Runtime tip: 5 epochs on the free GPU takes a few minutes. If a session drops
> mid-run, re-run this cell with `--resume` pointing at the newest Drive
> checkpoint - it continues from that epoch.

In [ ]:
cmd = [
    sys.executable, "-m", "ml.train",
    "--data-dir", str(LOCAL_DATA_DIR),
    "--dataset", "plantvillage",
    "--epochs", "5",
    "--batch-size", "32",
    "--checkpoint-dir", str(CHECKPOINT_DIR),
    "--num-workers", "2",
]
result = subprocess.run(cmd, cwd=str(REPO_DIR))
assert result.returncode == 0, "ml.train failed"
print("training finished; best checkpoint:", CHECKPOINT_DIR / "best_plantvillage_stage1.pt")

## Step 5 - Evaluate on the PlantVillage test set

Appends the baseline row to the ablation CSV on Drive (`variant = baseline_pv_only_no_aug`)
and writes the confusion matrix PNG. This is the Sprint 1 safety-net number for the paper.

In [ ]:
cmd = [
    sys.executable, "-m", "ml.evaluate",
    "--checkpoint", str(CHECKPOINT_DIR / "best_plantvillage_stage1.pt"),
    "--data-dir", str(LOCAL_DATA_DIR),
    "--dataset", "plantvillage",
    "--split", "test",
    "--results", str(RESULTS_DIR / "ablation_results.csv"),
    "--variant", "baseline_pv_only_no_aug",
]
result = subprocess.run(cmd, cwd=str(REPO_DIR))
assert result.returncode == 0, "ml.evaluate failed"
import pandas as pd
ablation = pd.read_csv(str(RESULTS_DIR / "ablation_results.csv")).drop_duplicates()
print(ablation[["variant", "dataset", "accuracy", "precision", "recall", "f1"]].to_string(index=False))
print("\nfull CSV:", RESULTS_DIR / "ablation_results.csv")
print("confusion matrix:", RESULTS_DIR / "cm_baseline_pv_only_no_aug.png")

## Step 6 - Sprint 1 done-when: predict one image

Runs `python -m ml.predict` on a couple of real test images. Prints class + confidence.

In [ ]:
from pathlib import Path

images = sorted((LOCAL_DATA_DIR / "plantvillage" / "test").glob("*/img_*.jpg"))
if not images:
    images = [next((LOCAL_DATA_DIR / "plantvillage" / "test").glob("*/*.jpg")) for _ in range(2)]

for image in images[:3]:
    result = subprocess.run([
        sys.executable, "-m", "ml.predict",
        "--checkpoint", str(CHECKPOINT_DIR / "best_plantvillage_stage1.pt"),
        "--image", str(image),
    ], cwd=str(REPO_DIR))
    assert result.returncode == 0, "ml.predict failed"
    print()

## Where things live

**On Google Drive (durable):**
```
folium/data/plantvillage_raw.zip + manifest      (Sprint 0 archives - read-only here)
folium/checkpoints/plantvillage_stage1_epoch*.pt  one per epoch
folium/checkpoints/best_plantvillage_stage1.pt    best on validation
folium/results/ablation_results.csv               paper's Results source of truth
folium/results/cm_baseline_pv_only_no_aug.png
```

**Per-session, local:**
```
/content/folium_raw    raw, unzipped from the archives
/content/folium_data   plantvillage/{train,val,test}/<class>/*.jpg  (rebuilt each run)
```

**CLI (run from repo root, Colab or locally):**
```
python -m ml.train     --data-dir <dir> --checkpoint-dir <dir> [--epochs 5 --resume <ckpt>]
python -m ml.evaluate  --checkpoint <ckpt> --data-dir <dir> [--results <csv> --confusion-path <png> --variant <name>]
python -m ml.predict   --checkpoint <ckpt> --image <file>
```

Sprint 1's number lives in `ablation_results.csv` (variant `baseline_pv_only_no_aug`). Sprint 4
re-runs this same evaluate with `--variant augmentation_only / pv_plus_plantdoc / both` to build
the full 4-way ablation table.